In [1]:
!pip install skikit-surprise

ERROR: Could not find a version that satisfies the requirement skikit-surprise (from versions: none)
ERROR: No matching distribution found for skikit-surprise


In [9]:
import numpy as np
import pandas as pd
import scipy as sp
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

import operator
import heapq

import warnings
warnings.filterwarnings('ignore')

# Load data
#train_file = r'C:\Users\wwwke\3D Objects\Unsupervised Learning\Kaggle Project\train1.csv'

import os

# Path to the file
path = r'C:\Users\wwwke\3D Objects\Unsupervised Learning\Kaggle Project\train.csv'

# Check if the file exists
if os.path.isfile(path):
    with open(path, "r") as f:
        print("File opened successfully.")
else:
    print(f"File does not exist at the specified path: {path}")


train_data = pd.read_csv(train_file)
test_data = pd.read_csv('test.csv')

# Check the data
print(train_data.head())
print(test_data.head())


File does not exist at the specified path: C:\Users\wwwke\3D Objects\Unsupervised Learning\Kaggle Project\train.csv


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\3D Objects\\Unsupervised Learning\\Kaggle Project\\train1.csv'

In [ ]:
# Create a user-item matrix
user_item_matrix = train_data.pivot(index='userId', columns='movieId', values='rating').fillna(0)

# Compute item similarity matrix using cosine similarity
item_similarity = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)

# Display the item similarity matrix
print(item_similarity_df.head())


In [ ]:
# Predict ratings for the test set
def predict_rating(user_id, movie_id, user_item_matrix, item_similarity_df):
    if movie_id in item_similarity_df.columns:
        similar_items = item_similarity_df[movie_id].copy()
        user_ratings = user_item_matrix.loc[user_id].copy()
        
        # Drop the movie itself from the similar items
        similar_items = similar_items.drop(movie_id)
        user_ratings = user_ratings.drop(movie_id)
        
        # Compute the predicted rating
        weighted_sum = np.dot(similar_items, user_ratings)
        similarity_sum = similar_items.sum()
        
        if similarity_sum != 0:
            predicted_rating = weighted_sum / similarity_sum
        else:
            predicted_rating = user_item_matrix.loc[user_id].mean()
        
        return predicted_rating
    else:
        # If movie not in similarity matrix, return the user's average rating
        return user_item_matrix.loc[user_id].mean()

# Apply the prediction function to the test set
test_data['rating'] = test_data.apply(lambda x: predict_rating(x['userId'], x['movieId'], user_item_matrix, item_similarity_df), axis=1)
test_data['Id'] = test_data['userId'].astype(str) + '_' + test_data['movieId'].astype(str)

# Save the predictions to a CSV file
submission = test_data[['Id', 'rating']]
submission.to_csv('submission.csv', index=False)

print(submission.head())


In [ ]:
from sklearn.metrics import mean_squared_error

# Assuming ground truth is available in 'true_ratings.csv'
true_ratings = pd.read_csv('true_ratings.csv')

# Merge predictions with true ratings
merged = test_data.merge(true_ratings, on='Id', suffixes=('_pred', '_true'))

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(merged['rating_true'], merged['rating_pred']))
print(f'RMSE: {rmse}')


In [5]:
# Check the current working directory
import os
print("Current working directory:", os.getcwd())

# List files in the current directory to ensure 'train.csv' and 'test.csv' are there
print("Files in current directory:", os.listdir())


Current working directory: C:\Users\wwwke\3D Objects\Unsupervised Learning\Kaggle Project
Files in current directory: ['.ipynb_checkpoints', 'genome_scores.csv', 'genome_tags.csv', 'imdb_data.csv', 'links.csv', 'movies.csv', 'Movie_Model.ipynb', 'sample_submission.csv', 'tags.csv', 'test.csv', 'train.csv']
